# AI Trading Signal Bot — Backtest Analysis

**Course:** JEM207 — Data Processing in Python  
**Author:** Josef Hlahůlek  
**Date:** June 2026

---

This notebook analyses the performance of the AI trading signal bot over a historical backtest.

The bot uses a two-stage pipeline:
1. **Technical scanner** — identifies potential setups using EMA pullbacks, breakouts, and volume absorption patterns
2. **Claude AI (Anthropic)** — receives full market context and decides whether to enter or skip

Backtest data: `data/backtest_results_v4.csv` — 48 scanner triggers across 4 instruments.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
from pathlib import Path

# Matplotlib style
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#333',
    'axes.labelcolor': '#ccc',
    'xtick.color': '#aaa',
    'ytick.color': '#aaa',
    'text.color': '#ddd',
    'grid.color': '#2a2d3a',
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
    'legend.facecolor': '#1a1d27',
    'legend.edgecolor': '#333',
    'figure.figsize': (12, 5),
    'font.size': 11,
})

COLOR_WIN  = '#26c281'
COLOR_LOSS = '#e74c3c'
COLOR_SKIP = '#5b6eae'
COLOR_BLUE = '#4a90d9'
COLOR_ORANGE = '#f39c12'

print('Libraries loaded.')

## 1. Load and Inspect Data

In [ ]:
# Load backtest CSV
data_path = Path('..') / 'data' / 'backtest_results_v4.csv'
df = pd.read_csv(data_path, parse_dates=['trigger_ts'])

# Separate enters from skips
enters = df[df['decision'] == 'enter'].copy()
skips  = df[df['decision'] == 'skip'].copy()

print(f'Total scanner triggers: {len(df)}')
print(f'Claude ENTER decisions:  {len(enters)} ({len(enters)/len(df):.0%})')
print(f'Claude SKIP decisions:   {len(skips)} ({len(skips)/len(df):.0%})')
print(f'\nInstruments: {", ".join(sorted(df["symbol"].unique()))}')
print(f'Date range: {df["trigger_ts"].min().date()} → {df["trigger_ts"].max().date()}')
print()

df.head(8)

In [ ]:
# Summary statistics for entered trades
wins   = enters[enters['outcome'] == 'hit_tp']
losses = enters[enters['outcome'] == 'hit_sl']

print('=== Entered trades ===')
print(f'Win rate:        {len(wins)/len(enters):.1%}  ({len(wins)} wins / {len(losses)} losses)')
print(f'Avg PnL:         {enters["pnl_pct"].mean():+.3f}%')
print(f'Avg win PnL:     {wins["pnl_pct"].mean():+.3f}%')
print(f'Avg loss PnL:    {losses["pnl_pct"].mean():+.3f}%')
print(f'Avg hold time:   {enters["exit_hours"].mean():.1f} hours')
print(f'Avg confidence:  {enters["confidence"].mean():.1f}/10')
print(f'Avg tokens_in:   {enters["tokens_in"].mean():.0f} (per Claude call)')

## 2. Decision Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: overall enter vs skip pie
ax = axes[0]
labels = ['Enter', 'Skip']
sizes  = [len(enters), len(skips)]
colors = [COLOR_WIN, COLOR_SKIP]
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors,
    autopct='%1.0f%%', startangle=90,
    textprops={'color': '#ddd', 'fontsize': 13},
    wedgeprops={'linewidth': 1, 'edgecolor': '#0f1117'},
)
for at in autotexts:
    at.set_fontsize(14)
    at.set_fontweight('bold')
ax.set_title('Claude Decisions\n(all 48 triggers)', fontsize=13, color='#ddd', pad=15)

# Right: outcome for entered trades
ax = axes[1]
labels = ['Win (TP hit)', 'Loss (SL hit)']
sizes  = [len(wins), len(losses)]
colors = [COLOR_WIN, COLOR_LOSS]
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors,
    autopct='%1.0f%%', startangle=90,
    textprops={'color': '#ddd', 'fontsize': 13},
    wedgeprops={'linewidth': 1, 'edgecolor': '#0f1117'},
)
for at in autotexts:
    at.set_fontsize(14)
    at.set_fontweight('bold')
ax.set_title('Trade Outcomes\n(17 entered trades)', fontsize=13, color='#ddd', pad=15)

plt.suptitle('AI Signal Bot — Backtest Overview', fontsize=15, color='white', y=1.02)
plt.tight_layout()
plt.savefig('decision_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 3. Per-Instrument Breakdown

In [ ]:
# Per-symbol stats
symbol_stats = []
for sym, grp in df.groupby('symbol'):
    e = grp[grp['decision'] == 'enter']
    w = e[e['outcome'] == 'hit_tp']
    l = e[e['outcome'] == 'hit_sl']
    symbol_stats.append({
        'Symbol': sym,
        'Triggers': len(grp),
        'Enters': len(e),
        'Skips': len(grp) - len(e),
        'Wins': len(w),
        'Losses': len(l),
        'Win Rate': f'{len(w)/len(e):.0%}' if len(e) > 0 else 'N/A',
        'Avg PnL%': f"{e['pnl_pct'].mean():+.2f}%" if len(e) > 0 else 'N/A',
    })

sym_df = pd.DataFrame(symbol_stats)
print(sym_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

symbols = sorted(df['symbol'].unique())
x = np.arange(len(symbols))
width = 0.3

# Left: triggers, enters, skips per symbol
ax = axes[0]
by_sym = df.groupby('symbol')['decision'].value_counts().unstack(fill_value=0)
by_sym = by_sym.reindex(sorted(by_sym.index))
enter_counts = by_sym.get('enter', pd.Series(0, index=by_sym.index))
skip_counts  = by_sym.get('skip', pd.Series(0, index=by_sym.index))
ax.bar(x - width/2, enter_counts.values, width, label='Enter', color=COLOR_WIN, alpha=0.85)
ax.bar(x + width/2, skip_counts.values,  width, label='Skip',  color=COLOR_SKIP, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(by_sym.index, rotation=15)
ax.set_ylabel('Count')
ax.set_title('Triggers per Instrument', fontsize=13)
ax.legend()
ax.grid(axis='y')

# Right: win/loss per symbol (entered trades only)
ax = axes[1]
win_by_sym  = enters[enters['outcome']=='hit_tp'].groupby('symbol').size()
loss_by_sym = enters[enters['outcome']=='hit_sl'].groupby('symbol').size()
syms_e = sorted(enters['symbol'].unique())
xe = np.arange(len(syms_e))
ax.bar(xe - width/2, [win_by_sym.get(s, 0) for s in syms_e],  width, label='Win (TP)', color=COLOR_WIN, alpha=0.85)
ax.bar(xe + width/2, [loss_by_sym.get(s, 0) for s in syms_e], width, label='Loss (SL)', color=COLOR_LOSS, alpha=0.85)
ax.set_xticks(xe)
ax.set_xticklabels(syms_e, rotation=15)
ax.set_ylabel('Count')
ax.set_title('Wins vs Losses per Instrument', fontsize=13)
ax.legend()
ax.grid(axis='y')

plt.suptitle('Per-Instrument Breakdown', fontsize=14, color='white')
plt.tight_layout()
plt.savefig('per_instrument.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 4. PnL Distribution and Cumulative Returns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: PnL histogram
ax = axes[0]
bins = np.linspace(enters['pnl_pct'].min() - 0.1, enters['pnl_pct'].max() + 0.1, 20)
ax.hist(wins['pnl_pct'],   bins=bins, color=COLOR_WIN,  alpha=0.75, label='Win (TP hit)',  edgecolor='#0f1117')
ax.hist(losses['pnl_pct'], bins=bins, color=COLOR_LOSS, alpha=0.75, label='Loss (SL hit)', edgecolor='#0f1117')
ax.axvline(enters['pnl_pct'].mean(), color=COLOR_ORANGE, linestyle='--', lw=2,
           label=f'Mean {enters["pnl_pct"].mean():+.2f}%')
ax.axvline(0, color='white', linestyle=':', lw=1, alpha=0.5)
ax.set_xlabel('PnL %')
ax.set_ylabel('Count')
ax.set_title('PnL Distribution (entered trades)', fontsize=13)
ax.legend()
ax.grid(axis='y')

# Right: cumulative PnL over time
ax = axes[1]
enters_sorted = enters.sort_values('trigger_ts')
cum_pnl = enters_sorted['pnl_pct'].cumsum()
colors_bar = [COLOR_WIN if p > 0 else COLOR_LOSS for p in enters_sorted['pnl_pct']]
ax.bar(range(len(enters_sorted)), enters_sorted['pnl_pct'], color=colors_bar, alpha=0.7, edgecolor='#0f1117')
ax2 = ax.twinx()
ax2.plot(range(len(enters_sorted)), cum_pnl, color=COLOR_BLUE, lw=2.5, label='Cumulative PnL')
ax2.set_ylabel('Cumulative PnL %', color=COLOR_BLUE)
ax2.tick_params(axis='y', colors=COLOR_BLUE)
ax.set_xlabel('Trade #')
ax.set_ylabel('PnL per trade %')
ax.set_title('Trade-by-Trade PnL + Cumulative', fontsize=13)
ax.grid(axis='y')
ax2.legend(loc='upper left')

plt.suptitle('PnL Analysis', fontsize=14, color='white')
plt.tight_layout()
plt.savefig('pnl_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

print(f'Total cumulative PnL: {enters["pnl_pct"].sum():+.2f}%')
print(f'Best trade:  {enters["pnl_pct"].max():+.2f}%')
print(f'Worst trade: {enters["pnl_pct"].min():+.2f}%')

## 5. Scanner Filter Analysis

In [ ]:
# Performance by filter
filter_stats = []
for fname, grp in df.groupby('filter_name'):
    e = grp[grp['decision'] == 'enter']
    w = e[e['outcome'] == 'hit_tp']
    filter_stats.append({
        'Filter': fname,
        'Triggers': len(grp),
        'Enter rate': f'{len(e)/len(grp):.0%}',
        'Win rate': f'{len(w)/len(e):.0%}' if len(e) > 0 else 'N/A',
        'Avg PnL%': f"{e['pnl_pct'].mean():+.3f}%" if len(e) > 0 else 'N/A',
    })

filt_df = pd.DataFrame(filter_stats)
print(filt_df.to_string(index=False))
print()
print('Filters: ema_pullback = pullback to EMA20, breakout_atr = ATR breakout, volume_absorption = volume spike')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
filter_names = sorted(df['filter_name'].unique())
x = np.arange(len(filter_names))

# Left: trigger count by filter
ax = axes[0]
counts_e = [len(df[(df['filter_name']==f) & (df['decision']=='enter')]) for f in filter_names]
counts_s = [len(df[(df['filter_name']==f) & (df['decision']=='skip')]) for f in filter_names]
ax.bar(x, counts_e, label='Enter', color=COLOR_WIN, alpha=0.85, edgecolor='#0f1117')
ax.bar(x, counts_s, bottom=counts_e, label='Skip', color=COLOR_SKIP, alpha=0.85, edgecolor='#0f1117')
ax.set_xticks(x)
ax.set_xticklabels([f.replace('_', '\n') for f in filter_names], fontsize=10)
ax.set_ylabel('Triggers')
ax.set_title('Scanner Filter — Trigger Counts', fontsize=13)
ax.legend()
ax.grid(axis='y')

# Right: avg PnL per filter (entered only)
ax = axes[1]
avg_pnls = []
for f in filter_names:
    subset = enters[enters['filter_name'] == f]['pnl_pct']
    avg_pnls.append(subset.mean() if len(subset) > 0 else 0)
colors_f = [COLOR_WIN if p >= 0 else COLOR_LOSS for p in avg_pnls]
bars = ax.bar(x, avg_pnls, color=colors_f, alpha=0.85, edgecolor='#0f1117')
ax.axhline(0, color='white', linestyle=':', lw=1, alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels([f.replace('_', '\n') for f in filter_names], fontsize=10)
ax.set_ylabel('Average PnL %')
ax.set_title('Avg PnL per Filter (entered trades)', fontsize=13)
ax.grid(axis='y')
for bar, val in zip(bars, avg_pnls):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:+.2f}%', ha='center', va='bottom', fontsize=11, color='#ddd')

plt.suptitle('Scanner Filter Performance', fontsize=14, color='white')
plt.tight_layout()
plt.savefig('filter_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 6. Confidence Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: confidence distribution for enters vs skips
ax = axes[0]
conf_vals = sorted(df['confidence'].unique())
x = np.arange(len(conf_vals))
width = 0.35
e_conf = [len(enters[enters['confidence']==c]) for c in conf_vals]
s_conf = [len(skips[skips['confidence']==c]) for c in conf_vals]
ax.bar(x - width/2, e_conf, width, label='Enter', color=COLOR_WIN, alpha=0.85, edgecolor='#0f1117')
ax.bar(x + width/2, s_conf, width, label='Skip',  color=COLOR_SKIP, alpha=0.85, edgecolor='#0f1117')
ax.set_xticks(x)
ax.set_xticklabels([f'{c}/10' for c in conf_vals])
ax.set_ylabel('Count')
ax.set_title('Confidence Distribution\n(enter vs skip)', fontsize=13)
ax.legend()
ax.grid(axis='y')
ax.axvline(1.5, color='white', linestyle='--', lw=1.5, alpha=0.6, label='Min threshold (6/10)')
ax.text(1.6, max(max(e_conf), max(s_conf))*0.95, 'Min\nthreshold', color='#aaa', fontsize=9)

# Right: win rate by confidence level
ax = axes[1]
enter_conf_vals = sorted(enters['confidence'].unique())
win_rates = []
for c in enter_conf_vals:
    subset = enters[enters['confidence'] == c]
    wr = len(subset[subset['outcome']=='hit_tp']) / len(subset) if len(subset) > 0 else 0
    win_rates.append(wr)
colors_wr = [COLOR_WIN if w >= 0.5 else COLOR_LOSS for w in win_rates]
bars = ax.bar(enter_conf_vals, win_rates, color=colors_wr, alpha=0.85, edgecolor='#0f1117', width=0.5)
ax.axhline(0.5, color='white', linestyle='--', lw=1.5, alpha=0.6, label='50% threshold')
ax.set_xlabel('Claude Confidence')
ax.set_ylabel('Win Rate')
ax.set_title('Win Rate by Confidence Level', fontsize=13)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend()
ax.grid(axis='y')
for bar, val in zip(bars, win_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.0%}', ha='center', va='bottom', fontsize=12, color='#ddd')

plt.suptitle('Claude Confidence vs Performance', fontsize=14, color='white')
plt.tight_layout()
plt.savefig('confidence_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 7. Hold Time Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: hold time histogram
ax = axes[0]
ax.hist(wins['exit_hours'],   bins=10, color=COLOR_WIN,  alpha=0.75, label='Win', edgecolor='#0f1117')
ax.hist(losses['exit_hours'], bins=10, color=COLOR_LOSS, alpha=0.75, label='Loss', edgecolor='#0f1117')
ax.axvline(12, color=COLOR_ORANGE, linestyle='--', lw=2, label='12h time-exit rule')
ax.set_xlabel('Hold Time (hours)')
ax.set_ylabel('Count')
ax.set_title('Hold Time Distribution', fontsize=13)
ax.legend()
ax.grid(axis='y')

# Right: hold time vs pnl scatter
ax = axes[1]
colors_scatter = [COLOR_WIN if o == 'hit_tp' else COLOR_LOSS for o in enters['outcome']]
ax.scatter(enters['exit_hours'], enters['pnl_pct'],
           c=colors_scatter, s=80, alpha=0.85, edgecolors='#0f1117', lw=0.5)
ax.axvline(12, color=COLOR_ORANGE, linestyle='--', lw=2, alpha=0.8, label='12h time-exit')
ax.axhline(0, color='white', linestyle=':', lw=1, alpha=0.4)
ax.set_xlabel('Hold Time (hours)')
ax.set_ylabel('PnL %')
ax.set_title('Hold Time vs PnL', fontsize=13)
ax.legend(handles=[
    plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=COLOR_WIN,  markersize=10, label='Win'),
    plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=COLOR_LOSS, markersize=10, label='Loss'),
    plt.Line2D([0],[0], color=COLOR_ORANGE, linestyle='--', label='12h rule'),
])
ax.grid()

plt.suptitle('Trade Hold Time Analysis', fontsize=14, color='white')
plt.tight_layout()
plt.savefig('hold_time.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

print(f'Avg win hold time:  {wins["exit_hours"].mean():.1f}h')
print(f'Avg loss hold time: {losses["exit_hours"].mean():.1f}h')
print(f'Trades hitting 12h: {len(enters[enters["exit_hours"] >= 12])}')

## 8. Claude API Token Usage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: tokens_in per decision type
ax = axes[0]
ax.scatter(range(len(enters)), enters.sort_values('trigger_ts')['tokens_in'],
           c=COLOR_WIN, s=60, alpha=0.8, label='Enter', edgecolors='#0f1117', lw=0.5)
ax.scatter(range(len(skips)), skips.sort_values('trigger_ts')['tokens_in'],
           c=COLOR_SKIP, s=60, alpha=0.8, label='Skip', edgecolors='#0f1117', lw=0.5)
ax.axhline(df['tokens_in'].mean(), color=COLOR_ORANGE, linestyle='--', lw=2,
           label=f'Mean {df["tokens_in"].mean():.0f} tokens')
ax.set_xlabel('Call #')
ax.set_ylabel('Tokens in')
ax.set_title('Claude Context Size per Call', fontsize=13)
ax.legend()
ax.grid()

# Right: tokens_out (response length)
ax = axes[1]
ax.hist(enters['tokens_out'], bins=12, color=COLOR_WIN,  alpha=0.75, label='Enter', edgecolor='#0f1117')
ax.hist(skips['tokens_out'],  bins=12, color=COLOR_SKIP, alpha=0.75, label='Skip',  edgecolor='#0f1117')
ax.set_xlabel('Tokens out (Claude response)')
ax.set_ylabel('Count')
ax.set_title('Claude Response Length Distribution', fontsize=13)
ax.legend()
ax.grid(axis='y')

plt.suptitle('Claude API Token Usage', fontsize=14, color='white')
plt.tight_layout()
plt.savefig('token_usage.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

total_tokens_in  = df['tokens_in'].sum()
total_tokens_out = df['tokens_out'].sum()
# Approximate cost: claude-3-5-haiku input $0.80/M, output $4/M
# With 90% prompt cache hit: effective input cost = 0.1 * 0.80 + 0.9 * 0.08 = 0.152
cost_no_cache    = total_tokens_in * 0.80 / 1e6 + total_tokens_out * 4 / 1e6
cost_with_cache  = total_tokens_in * 0.152 / 1e6 + total_tokens_out * 4 / 1e6

print(f'Total tokens in:  {total_tokens_in:,}')
print(f'Total tokens out: {total_tokens_out:,}')
print(f'Avg tokens in per call:  {df["tokens_in"].mean():.0f}')
print(f'Avg tokens out per call: {df["tokens_out"].mean():.0f}')
print(f'Estimated cost without cache: ${cost_no_cache:.4f}')
print(f'Estimated cost with 90% cache: ${cost_with_cache:.4f}')
print(f'Cache savings: {1 - cost_with_cache/cost_no_cache:.0%}')

## 9. Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(16, 9))
fig.patch.set_facecolor('#0f1117')

# Title
fig.text(0.5, 0.97, 'AI Signal Bot — Backtest Summary Dashboard',
         ha='center', va='top', fontsize=18, color='white', fontweight='bold')
fig.text(0.5, 0.93, f'JEM207 | {len(df)} triggers | {len(enters)} trades | """.join(sorted(df["symbol"].unique()))',
         ha='center', va='top', fontsize=12, color='#aaa')

# Key metrics as text
metrics = [
    ('Scanner\nTriggers', str(len(df)), '#4a90d9'),
    ('Claude\nEnter', f'{len(enters)}\n({len(enters)/len(df):.0%})', COLOR_WIN),
    ('Win Rate', f'{len(wins)/len(enters):.0%}', COLOR_WIN if len(wins)/len(enters) > 0.5 else COLOR_LOSS),
    ('Avg PnL', f'{enters["pnl_pct"].mean():+.2f}%', COLOR_WIN if enters['pnl_pct'].mean() > 0 else COLOR_LOSS),
    ('Avg Hold', f'{enters["exit_hours"].mean():.1f}h', '#f39c12'),
    ('Avg Conf', f'{enters["confidence"].mean():.1f}/10', '#9b59b6'),
]

for i, (label, value, color) in enumerate(metrics):
    x = 0.09 + i * 0.145
    fig.text(x, 0.83, value, ha='center', fontsize=20, color=color, fontweight='bold')
    fig.text(x, 0.77, label, ha='center', fontsize=9, color='#aaa')

# Cumulative PnL plot
ax1 = fig.add_axes([0.05, 0.12, 0.55, 0.55])
ax1.set_facecolor('#1a1d27')
enters_s = enters.sort_values('trigger_ts')
cum = enters_s['pnl_pct'].cumsum()
ax1.fill_between(range(len(enters_s)), 0, cum, 
                  where=(cum >= 0), color=COLOR_WIN, alpha=0.3)
ax1.fill_between(range(len(enters_s)), 0, cum,
                  where=(cum < 0), color=COLOR_LOSS, alpha=0.3)
ax1.plot(range(len(enters_s)), cum, color=COLOR_BLUE, lw=2.5)
ax1.scatter(range(len(enters_s)),
            [c if o=='hit_tp' else c for c, o in zip(cum, enters_s['outcome'])],
            c=[COLOR_WIN if o=='hit_tp' else COLOR_LOSS for o in enters_s['outcome']],
            s=60, zorder=5, edgecolors='#0f1117', lw=0.5)
ax1.axhline(0, color='white', linestyle=':', lw=1, alpha=0.4)
ax1.set_xlabel('Trade #', color='#aaa')
ax1.set_ylabel('Cumulative PnL %', color='#aaa')
ax1.set_title('Cumulative PnL (all entered trades)', color='#ddd', fontsize=12)
ax1.grid()
ax1.tick_params(colors='#aaa')
for sp in ax1.spines.values(): sp.set_edgecolor('#333')

# Win/loss pie on right
ax2 = fig.add_axes([0.65, 0.35, 0.32, 0.35])
ax2.set_facecolor('#0f1117')
ax2.pie([len(wins), len(losses)],
        labels=['Win', 'Loss'],
        colors=[COLOR_WIN, COLOR_LOSS],
        autopct='%1.0f%%',
        startangle=90,
        textprops={'color': '#ddd', 'fontsize': 13},
        wedgeprops={'linewidth': 1.5, 'edgecolor': '#0f1117'})
ax2.set_title('Trade Outcomes', color='#ddd', fontsize=12, pad=10)

plt.savefig('dashboard.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 10. Conclusions

### Key Findings

**Claude's selectivity works:** The LLM filtered out 65% of scanner triggers (31 skips out of 48). This is by design — the scanner fires frequently on marginal setups, and Claude is meant to decline most of them.

**Win rate is near 50%:** With a 47% win rate and positive average PnL (+0.27%), the system shows a slight edge — because winners are held longer (TP distance) than losers (SL distance), the risk:reward ratio is positive even with sub-50% win rate.

**Confidence threshold matters:** Claude assigns confidence 6/10 or above only to trades it enters. The risk manager enforces a minimum confidence of 6, which correctly filters out low-conviction setups (scored 3–4).

**Filter quality varies:** The `ema_pullback` filter generates the most triggers and produces the best average PnL. `volume_absorption` has fewer triggers but higher selectivity.

**Prompt caching delivers ~80% token cost savings** — the large system prompt (8000+ tokens) is reused across calls, making the approach economically viable.

### Limitations

- Backtest covers only ~3 weeks of data (48 triggers) — statistically limited
- No slippage or commission modeled
- Pattern Day Trading (PDT) restrictions affected live results
- Claude's reasoning quality may vary with model updates

### Further Work

- Extend backtest to 6+ months across different market regimes
- Test on more instruments (SPY, QQQ, sector ETFs)
- Add news sentiment scoring as a separate feature
- Evaluate different confidence thresholds (5 vs 6 vs 7)